# Notebook 03 · Track B UNRATE forecasts with 2-year fixed windows and MASE

**Request.** Rerun results using the **Track B** dataset, **UNRATE**, a **2-year fixed window**, look-ahead horizons **1, 3, and 6 months**, and **Mean Absolute Scaled Error (MASE)** as the performance metric.

This notebook is deliberately explicit about assumptions because the phrase "2-year fixed window" can mean different things in forecasting. The implementation here uses a fixed 24-month **input/history window** for every supervised example. The train/test evaluation folds remain the shared project folds so these results are still comparable to other capstone notebooks. If the intended meaning was instead a 24-month rolling **training** window, that is a different experiment and should be run separately.


## 1 · Experiment assumptions

| Area | Assumption used in this notebook | Why / implication |
|---|---|---|
| Dataset | Use `data/processed/track_B_curated.parquet` as the feature source. | This is the project Track B dataset from `01_data_prep.ipynb`. |
| UNRATE feature | The saved Track B column `UNRATE` is a **stationary month-to-month change**, not the unemployment-rate level. This notebook renames it to `UNRATE_delta` for clarity. | Prevents confusing transformed features with the level target. |
| UNRATE level | Add `UNRATE_level` from `levels_panel.parquet` to the feature panel. | If we forecast the unemployment-rate level, the current level is known at forecast origin and is essential for a fair baseline/model comparison. |
| Target | Forecast the future **UNRATE level**: `UNRATE_level[t+h]`. | MASE is most interpretable for level forecasts. Forecasting the differenced stationary UNRATE series would make MASE less economically intuitive. |
| Horizons | `h = {1, 3, 6}` months. | Per request. |
| Fixed window | Each model input is the trailing 24 months of Track B features ending at forecast origin `t`. | This matches the project/quantum need for constant-length windows. |
| Flattening | The 24 × feature matrix is flattened into one vector for classical models. | Tree/SVR/kNN models consume tabular vectors, not 3D tensors. |
| Train/test folds | Use the existing shared expanding folds from `folds.json`. | Keeps comparability with prior notebooks. This is **not** a 24-month rolling training window. |
| Horizon leakage | For horizon `h`, remove the last `h` months of each training fold because their future labels would not yet be known. | Avoids training on unavailable future outcomes. |
| Overlapping test folds | Existing 5-year test blocks overlap. We keep the latest prediction for each `(model, horizon, date)`. | Avoids over-weighting repeated target dates. |
| Metric | Mean Absolute Scaled Error: `mean(abs(error) / scale)`, where `scale` is the in-fold mean absolute one-step change in the training UNRATE level. | MASE < 1 means the model beats the in-fold naive one-step scale on average. |
| Paper models | Use the same Konduri-inspired model set where meaningful: ARD, naive/random-walk, AdaBoost, Random Forest, AdaBoost-DI, kNN-DI, SVR linear-DI, XGBoost-DI. | These came from Konduri & Li Table 1 and the best-model discussion in Tables 2-8. |
| Hyperparameters | Fixed simple hyperparameters; no tuning on test data. | Avoids hidden leakage and makes the notebook transparent. |


## 2 · Load data and construct Track B + UNRATE level panel


In [1]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

TRACK_B_PATH = DATA_DIR / "track_B_curated.parquet"
LEVELS_PATH = DATA_DIR / "levels_panel.parquet"
FOLDS_PATH = DATA_DIR / "folds.json"

track_b_raw = pd.read_parquet(TRACK_B_PATH)
levels = pd.read_parquet(LEVELS_PATH)
track_b_raw.index = pd.to_datetime(track_b_raw.index)
levels.index = pd.to_datetime(levels.index)

# Rename the transformed/stationary UNRATE feature for clarity.
track_b = track_b_raw.rename(columns={"UNRATE": "UNRATE_delta"}).copy()
track_b["UNRATE_level"] = levels["UNRATE"].reindex(track_b.index)

# Keep deterministic column order.
front = ["UNRATE_level", "UNRATE_delta"]
track_b = track_b[front + [c for c in track_b.columns if c not in front]]

print("Track B raw columns     :", list(track_b_raw.columns))
print("Track B modeling columns:", list(track_b.columns))
print("Shape:", track_b.shape)
print("Date range:", track_b.index.min(), "to", track_b.index.max())
print("Missing cells:", int(track_b.isna().sum().sum()))
track_b.head()


Track B raw columns     : ['UNRATE', 'PERMIT', 'S&P 500', 'UMCSENTx', 'T10Y3M_level', 'T10Y3M_delta']
Track B modeling columns: ['UNRATE_level', 'UNRATE_delta', 'PERMIT', 'S&P 500', 'UMCSENTx', 'T10Y3M_level', 'T10Y3M_delta']
Shape: (767, 7)
Date range: 1962-04-01 00:00:00 to 2026-02-01 00:00:00
Missing cells: 1


,UNRATE_level,UNRATE_delta,PERMIT,S&P 500,UMCSENTx,T10Y3M_level,T10Y3M_delta
date,,,,,,,
1962-04-01,5.6,0.0,7.118826,-0.032387,0.0,1.11,NaN
1962-05-01,5.5,-0.1,7.040536,-0.077267,-4.5,1.18,0.07
1962-06-01,5.5,0.0,7.050989,-0.124253,0.0,1.18,0.00
1962-07-01,5.4,-0.1,7.080868,0.023802,0.0,1.09,-0.09
1962-08-01,5.7,0.3,7.090077,0.026844,-3.8,1.16,0.07


In [2]:
with open(FOLDS_PATH) as f:
    folds_json = json.load(f)["shared"]

folds = []
for i, f in enumerate(folds_json):
    train_idx = track_b.loc[f["train_start"]:f["train_end"]].index
    test_idx = track_b.loc[f["test_start"]:f["test_end"]].index
    folds.append({**f, "fold_id": i, "train_idx": train_idx, "test_idx": test_idx})

WINDOW_MONTHS = 24
HORIZONS = [1, 3, 6]
TARGET = "UNRATE_level"
LEVEL_TARGET_COL = "UNRATE"
DEDUP_POLICY = "latest"
SEED = 42

print(f"Loaded {len(folds)} folds")
print("First fold:", folds[0]["train_start"], "→", folds[0]["train_end"], "/ test", folds[0]["test_start"], "→", folds[0]["test_end"])


Loaded 39 folds
First fold: 1962-04-01 → 1982-03-01 / test 1982-04-01 → 1987-03-01


## 3 · What the Track B features mean, and why they are relevant for UNRATE

Track B is the small, interpretable feature set that came from the sponsor-facing leading-indicator discussion in `01_data_prep.ipynb`. For this UNRATE experiment, I use the saved Track B features plus the current unemployment-rate level.

| Feature used here | What it means | Transformation / source in this project | Why it may help forecast UNRATE |
|---|---|---|---|
| `UNRATE_level` | Civilian unemployment rate, percent of labor force unemployed. | Pulled from `levels_panel.parquet`, not stationarized. This is the current level at forecast origin `t`. | Unemployment is highly persistent. The current level is the natural starting point for any 1-, 3-, or 6-month forecast and is what a naive/random-walk benchmark uses. |
| `UNRATE_delta` | Month-to-month change in the unemployment rate. | This is the original Track B `UNRATE` column after McCracken-Ng transformation code 2, renamed to avoid confusion. | Captures recent labor-market momentum. Rising unemployment tends to continue during slowdowns; falling unemployment can indicate ongoing expansion. |
| `PERMIT` | New private housing building permits. | Stationarized FRED-MD feature. In the saved Track B panel it is transformed according to the FRED-MD code rather than used as raw level. | Housing is cyclical and interest-rate sensitive. Permits often weaken before construction, employment, and broader labor demand weaken. |
| `S&P 500` | Broad equity-market index. | Stationarized FRED-MD feature, effectively an equity return/growth measure after transformation. | Equity markets can price expected future earnings and recession risk before labor-market data deteriorate. Useful as a forward-looking financial signal. |
| `UMCSENTx` | University of Michigan consumer sentiment index. | Stationarized FRED-MD feature. | Consumer sentiment falls when households expect weaker income/job prospects. It can lead consumption and hiring conditions. |
| `T10Y3M_level` | 10-year Treasury yield minus 3-month Treasury bill rate, in percentage points. | Constructed in preprocessing from raw levels: `GS10 - TB3MS`. Kept as a level because inversion is a state, not a change. | Yield-curve inversions are classic recession/labor-market warning signals. A low or negative spread can precede rising unemployment. |
| `T10Y3M_delta` | Month-to-month change in the 10-year minus 3-month yield spread. | Difference of `T10Y3M_level`. | Captures whether financial conditions are steepening or flattening recently, which may add short-run timing information beyond the level. |

### Why this small feature set is defensible for UNRATE

- **Labor-market persistence:** `UNRATE_level` and `UNRATE_delta` encode where unemployment is now and whether it is moving.
- **Leading real-economy signal:** `PERMIT` proxies housing/construction-cycle weakness, which often shows up before labor-market deterioration.
- **Forward-looking financial signal:** `S&P 500` and `T10Y3M_level` summarize market expectations and monetary/term-structure conditions.
- **Household expectations:** `UMCSENTx` captures consumer confidence, which is linked to spending, business conditions, and hiring.
- **Interpretability:** Track B is small enough that every feature can be explained to a sponsor or professor, unlike the full 123-variable panel.

### Important caveat

Track B is **not** necessarily the statistically optimal feature set for UNRATE. It is an interpretable leading-indicator set. A full-panel or automated feature-selection model may perform better, but Track B is easier to defend and aligns with the project goal of transparent comparisons against quantum models that need low-dimensional inputs.


## 4 · Build fixed 24-month window features

For each forecast origin `t`, the feature vector contains all Track B columns for months `t-23` through `t`, flattened in chronological order. The target for horizon `h` is `UNRATE_level[t+h]`.


In [3]:
def build_fixed_window_features(df: pd.DataFrame, window: int) -> pd.DataFrame:
    '''Flatten trailing fixed windows. Row at t contains df[t-window+1 : t].'''
    frames = []
    for lag in range(window - 1, -1, -1):
        shifted = df.shift(lag)
        suffix = f"t-{lag}" if lag else "t"
        shifted.columns = [f"{col}__{suffix}" for col in df.columns]
        frames.append(shifted)
    return pd.concat(frames, axis=1)

X_window = build_fixed_window_features(track_b, WINDOW_MONTHS)
y_level = levels[LEVEL_TARGET_COL].reindex(track_b.index)

print("X_window shape:", X_window.shape)
print("Feature count =", WINDOW_MONTHS, "months ×", track_b.shape[1], "features =", X_window.shape[1])
X_window.dropna().head()


X_window shape: (767, 168)
Feature count = 24 months × 7 features = 168


,UNRATE_level__t-23,UNRATE_delta__t-23,PERMIT__t-23,S&P 500__t-23,UMCSENTx__t-23,T10Y3M_level__t-23,T10Y3M_delta__t-23,UNRATE_level__t-22,UNRATE_delta__t-22,PERMIT__t-22,...,UMCSENTx__t-1,T10Y3M_level__t-1,T10Y3M_delta__t-1,UNRATE_level__t,UNRATE_delta__t,PERMIT__t,S&P 500__t,UMCSENTx__t,T10Y3M_level__t,T10Y3M_delta__t
date,,,,,,,,,,,,,,,,,,,,,
1964-04-01,5.5,-0.1,7.040536,-0.077267,-4.5,1.18,0.07,5.5,0.0,7.050989,...,0.0,0.68,0.06,5.3,-0.1,7.142037,0.014363,0.0,0.76,0.08
1964-05-01,5.5,0.0,7.050989,-0.124253,0.0,1.18,0.00,5.4,-0.1,7.080868,...,0.0,0.76,0.08,5.1,-0.2,7.169350,0.009710,-1.0,0.72,-0.04
1964-06-01,5.4,-0.1,7.080868,0.023802,0.0,1.09,-0.09,5.7,0.3,7.090077,...,-1.0,0.72,-0.04,5.2,0.1,7.154615,-0.005964,0.0,0.69,-0.03
1964-07-01,5.7,0.3,7.090077,0.026844,-3.8,1.16,0.07,5.6,-0.1,7.109062,...,0.0,0.69,-0.03,4.9,-0.3,7.173192,0.036466,0.0,0.73,0.04
1964-08-01,5.6,-0.1,7.109062,-0.008926,0.0,1.20,0.04,5.4,-0.2,7.074117,...,0.0,0.73,0.04,5.0,0.1,7.174724,-0.014768,2.1,0.69,-0.04


## 5 · Models

- `ARD(6)`: linear autoregression using the latest six monthly UNRATE levels.
- `Naive / Random Walk`: predicts the current UNRATE level for every horizon.
- `AdaBoost` and `Random Forest`: use the flattened 24-month Track B window directly.
- `AdaBoost-DI`, `kNN-DI`, `SVR linear-DI`, `XGBoost-DI`: first reduce the flattened 24-month Track B window with training-fold PCA (`80%` variance), then fit the model.


In [4]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost unavailable:", repr(e))


def full_model_factories():
    return {
        "AdaBoost": AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=SEED),
            n_estimators=50,
            learning_rate=0.05,
            random_state=SEED,
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=80,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=SEED,
            n_jobs=-1,
        ),
    }


def di_model_factories():
    models = {
        "AdaBoost-DI": AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=SEED),
            n_estimators=50,
            learning_rate=0.05,
            random_state=SEED,
        ),
        "kNN-DI": Pipeline([
            ("scale", StandardScaler()),
            ("knn", KNeighborsRegressor(n_neighbors=10, weights="distance", metric="euclidean")),
        ]),
        "SVR linear-DI": TransformedTargetRegressor(
            regressor=Pipeline([
                ("scale", StandardScaler()),
                ("svr", SVR(kernel="linear", C=1.0, epsilon=0.05)),
            ]),
            transformer=StandardScaler(),
        ),
    }
    if HAS_XGB:
        models["XGBoost-DI"] = XGBRegressor(
            n_estimators=80,
            learning_rate=0.05,
            max_depth=3,
            min_child_weight=5,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=1,
            verbosity=0,
        )
    return models

MODEL_ORDER = ["ARD(6)", "Naive / Random Walk", "AdaBoost", "Random Forest", "AdaBoost-DI", "kNN-DI", "SVR linear-DI", "XGBoost-DI"]
print("Models:", MODEL_ORDER)


Models: ['ARD(6)', 'Naive / Random Walk', 'AdaBoost', 'Random Forest', 'AdaBoost-DI', 'kNN-DI', 'SVR linear-DI', 'XGBoost-DI']


In [5]:
def make_ard_features(y: pd.Series, lags: int = 6) -> pd.DataFrame:
    return pd.concat([y.shift(lag).rename(f"UNRATE_level_lag{lag}") for lag in range(lags)], axis=1)


def mase_scale(y_train_level: pd.Series) -> float:
    diffs = y_train_level.dropna().diff().abs().dropna()
    scale = float(diffs.mean())
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Invalid MASE scale; training series may be constant or empty.")
    return scale


def append_rows(rows, model, h, fold, dates, y_true, y_pred, feature_set, scale, extra=None):
    extra = extra or {}
    for dt, yt, yp in zip(dates, y_true, y_pred):
        abs_err = abs(float(yt) - float(yp))
        rows.append({
            "model": model,
            "target": TARGET,
            "h": h,
            "fold_id": fold["fold_id"],
            "fold_train_end": fold["train_end"],
            "date": dt,
            "y_true": float(yt),
            "y_pred": float(yp),
            "abs_error": abs_err,
            "mase_scale": scale,
            "scaled_abs_error": abs_err / scale,
            "feature_set": feature_set,
            **extra,
        })


def valid_xy(X: pd.DataFrame, y: pd.Series, idx):
    Xb = X.reindex(idx)
    yb = y.reindex(idx)
    mask = Xb.notna().all(axis=1) & yb.notna()
    good_idx = idx[mask.values]
    return good_idx, X.loc[good_idx], y.loc[good_idx]


## 6 · Run forecasts


In [6]:
def fit_predict_one(h: int, fold: dict) -> list[dict]:
    y_h = y_level.shift(-h)
    train_idx = fold["train_idx"]
    test_idx = fold["test_idx"]
    train_end = pd.Timestamp(fold["train_end"])
    supervised_train_idx = train_idx[train_idx <= train_end - pd.DateOffset(months=h)]
    test_eval_idx = y_h.reindex(test_idx).dropna().index
    scale = mase_scale(y_level.reindex(supervised_train_idx))
    rows = []

    # ARD(6) on UNRATE levels.
    X_ard = make_ard_features(y_level, lags=6)
    tr_idx, Xtr, ytr = valid_xy(X_ard, y_h, supervised_train_idx)
    te_idx, Xte, yte = valid_xy(X_ard, y_h, test_eval_idx)
    if len(tr_idx) > 10 and len(te_idx) > 0:
        model = LinearRegression()
        model.fit(Xtr, ytr)
        append_rows(rows, "ARD(6)", h, fold, te_idx, yte, model.predict(Xte), "unrate_level_lags", scale)

    # Naive / random-walk level forecast.
    naive_pred = y_level.reindex(test_eval_idx)
    valid = naive_pred.notna() & y_h.reindex(test_eval_idx).notna()
    naive_idx = test_eval_idx[valid.values]
    append_rows(rows, "Naive / Random Walk", h, fold, naive_idx, y_h.loc[naive_idx], naive_pred.loc[naive_idx], "current_unrate_level", scale)

    # Full Track B fixed-window models.
    tr_idx, Xtr, ytr = valid_xy(X_window, y_h, supervised_train_idx)
    te_idx, Xte, yte = valid_xy(X_window, y_h, test_eval_idx)
    if len(tr_idx) > 20 and len(te_idx) > 0:
        for name, estimator in full_model_factories().items():
            model = clone(estimator)
            model.fit(Xtr, ytr)
            append_rows(rows, name, h, fold, te_idx, yte, model.predict(Xte), "track_b_24m_window", scale, {"n_features": Xtr.shape[1]})

        # PCA-DI models fit PCA on training window features only.
        pca_pipe = Pipeline([
            ("scale", StandardScaler()),
            ("pca", PCA(n_components=0.80, svd_solver="full", random_state=SEED)),
        ])
        Xtr_di = pca_pipe.fit_transform(Xtr)
        Xte_di = pca_pipe.transform(Xte)
        n_di = int(pca_pipe.named_steps["pca"].n_components_)
        pca_var = float(pca_pipe.named_steps["pca"].explained_variance_ratio_.sum())
        for name, estimator in di_model_factories().items():
            model = clone(estimator)
            model.fit(Xtr_di, ytr)
            append_rows(rows, name, h, fold, te_idx, yte, model.predict(Xte_di), "track_b_24m_window_pca", scale, {"n_di": n_di, "pca_var": pca_var})

    return rows


In [7]:
start = time.time()
rows = []
for h in HORIZONS:
    for fold in folds:
        rows.extend(fit_predict_one(h, fold))
    print(f"finished h={h}, rows so far={len(rows):,}")

preds_raw = pd.DataFrame(rows)
preds_raw["date"] = pd.to_datetime(preds_raw["date"])
preds_raw.to_parquet(RESULTS_DIR / "track_b_unrate_24m_mase_predictions_raw.parquet", index=False)
print(f"Raw rows: {len(preds_raw):,}")
print(f"Elapsed: {(time.time() - start)/60:.2f} minutes")
preds_raw.head()


finished h=1, rows so far=18,720


finished h=3, rows so far=37,440


finished h=6, rows so far=56,160
Raw rows: 56,160
Elapsed: 1.61 minutes


,model,target,h,fold_id,fold_train_end,date,y_true,y_pred,abs_error,mase_scale,scaled_abs_error,feature_set,n_features,n_di,pca_var
0,ARD(6),UNRATE_level,1,0,1982-03-01,1982-04-01,9.4,9.358845,0.041155,0.138235,0.297720,unrate_level_lags,NaN,NaN,NaN
1,ARD(6),UNRATE_level,1,0,1982-03-01,1982-05-01,9.6,9.516219,0.083781,0.138235,0.606074,unrate_level_lags,NaN,NaN,NaN
2,ARD(6),UNRATE_level,1,0,1982-03-01,1982-06-01,9.8,9.642714,0.157286,0.138235,1.137815,unrate_level_lags,NaN,NaN,NaN
3,ARD(6),UNRATE_level,1,0,1982-03-01,1982-07-01,9.8,9.897473,0.097473,0.138235,0.705124,unrate_level_lags,NaN,NaN,NaN
4,ARD(6),UNRATE_level,1,0,1982-03-01,1982-08-01,10.1,9.837945,0.262055,0.138235,1.895715,unrate_level_lags,NaN,NaN,NaN


In [8]:
# Deduplicate overlapping test windows.
preds = preds_raw.copy()
preds["fold_train_end"] = pd.to_datetime(preds["fold_train_end"])
preds = preds.sort_values(["model", "h", "date", "fold_train_end"])
if DEDUP_POLICY == "latest":
    preds = preds.drop_duplicates(["model", "h", "date"], keep="last")
elif DEDUP_POLICY == "earliest":
    preds = preds.drop_duplicates(["model", "h", "date"], keep="first")
else:
    raise ValueError(DEDUP_POLICY)

preds.to_parquet(RESULTS_DIR / "track_b_unrate_24m_mase_predictions.parquet", index=False)
print(f"Deduplicated rows: {len(preds):,}")
print(preds.groupby("model").size().reindex(MODEL_ORDER).dropna().astype(int))


Deduplicated rows: 12,384
model
ARD(6)                 1548
Naive / Random Walk    1548
AdaBoost               1548
Random Forest          1548
AdaBoost-DI            1548
kNN-DI                 1548
SVR linear-DI          1548
XGBoost-DI             1548
dtype: int64


## 7 · MASE results

Lower is better. `MASE < 1` means the model's mean absolute error is smaller than the in-fold naive one-step scaling error.


In [9]:
def summarize(pred_df: pd.DataFrame) -> pd.DataFrame:
    recs = []
    for (h, model), block in pred_df.groupby(["h", "model"]):
        recs.append({
            "target": TARGET,
            "h": h,
            "model": model,
            "n": len(block),
            "mae": float(block["abs_error"].mean()),
            "mase": float(block["scaled_abs_error"].mean()),
        })
    return pd.DataFrame(recs).sort_values(["h", "mase"])

metrics = summarize(preds)
metrics.to_csv(RESULTS_DIR / "track_b_unrate_24m_mase_metrics.csv", index=False)

mase_table = metrics.pivot_table(index="model", columns="h", values="mase").reindex(MODEL_ORDER)
mase_table.style.format("{:.3f}").background_gradient(axis=None, cmap="RdYlGn_r")


h,1,3,6
model,,,
ARD(6),1.256,2.440,3.774
Naive / Random Walk,1.185,2.424,4.020
AdaBoost,2.225,3.309,5.223
Random Forest,3.468,4.285,5.543
AdaBoost-DI,6.167,6.821,7.504
kNN-DI,9.438,9.642,9.835
SVR linear-DI,2.349,2.920,3.938
XGBoost-DI,4.552,5.280,6.136


In [10]:
# Best model at each look-ahead horizon.
best = metrics.sort_values("mase").groupby("h").head(1).sort_values("h")
best.style.format({"mae": "{:.3f}", "mase": "{:.3f}"})


,target,h,model,n,mae,mase
3,UNRATE_level,1,Naive / Random Walk,516,0.153,1.185
11,UNRATE_level,3,Naive / Random Walk,516,0.313,2.424
16,UNRATE_level,6,ARD(6),516,0.486,3.774


In [11]:
# Compact ranking across horizons.
summary = (
    metrics.groupby("model")
    .agg(mean_mase=("mase", "mean"), median_mase=("mase", "median"), best_mase=("mase", "min"), cells=("mase", "size"))
    .sort_values("mean_mase")
)
summary.style.format({"mean_mase": "{:.3f}", "median_mase": "{:.3f}", "best_mase": "{:.3f}"})


,mean_mase,median_mase,best_mase,cells
model,,,,
ARD(6),2.490,2.440,1.256,3
Naive / Random Walk,2.543,2.424,1.185,3
SVR linear-DI,3.069,2.920,2.349,3
AdaBoost,3.586,3.309,2.225,3
Random Forest,4.432,4.285,3.468,3
XGBoost-DI,5.322,5.280,4.552,3
AdaBoost-DI,6.831,6.821,6.167,3
kNN-DI,9.638,9.642,9.438,3


## 8 · Takeaways and grading notes

- This run uses **Track B + current UNRATE level** to forecast future UNRATE levels at 1, 3, and 6 months.
- The **2-year fixed window** is implemented as a 24-month feature/history window, not a 24-month training window.
- MASE is computed per prediction using the training-fold one-step UNRATE-level scale, then averaged.
- PCA for DI models is fit inside each fold only.
- Horizon leakage is avoided by purging the last `h` training months.
- Because test folds overlap, predictions are deduplicated before the final MASE table.
- If the class/professor expects a 24-month rolling training window instead, this notebook should be cloned and the fold generator changed rather than reusing `folds.json`.
